# 2MASS crossmatch

Crossmatch all stars in `training_stars.csv`, `validation_stars.csv`, `mmcoeval.csv`, and `cf_rotator_catalog.csv` to 2MASS to retrieve apparent Ks magnitude (`k_m`).

**Goal:** Compute absolute Ks magnitude `M_Ks`, then apply the Mann et al. (2019) M_Ks–M* relation to estimate stellar masses.

**Valid range for Mann 2019:** 4.5 < M_Ks < 10.5 (corresponding to 0.075–0.70 M☉)

**Columns retrieved from 2MASS:**
- `k_m` — apparent Ks magnitude
- `k_cmsig` — Ks magnitude uncertainty

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

CF_DATA = Path("cf_data")
sys.path.insert(0, "Mann_2019")  # make mk_mass importable

train = pd.read_csv(CF_DATA / "training_stars.csv", dtype={'source_id': 'Int64'})
val   = pd.read_csv(CF_DATA / "validation_stars.csv", dtype={'source_id': 'Int64'})
mm    = pd.read_csv(CF_DATA / "mmcoeval.csv", dtype={'source_id': 'Int64'})
cfrc  = pd.read_csv(CF_DATA / "cf_rotator_catalog.csv", dtype={'GaiaDR3_ID': 'Int64'})

print(f"training_stars:     {len(train)} rows")
print(f"validation_stars:   {len(val)} rows")
print(f"mmcoeval:           {len(mm)} rows")
print(f"cf_rotator_catalog: {len(cfrc)} rows")


training_stars:     4923 rows
validation_stars:   242 rows
mmcoeval:           34 rows
cf_rotator_catalog: 5832 rows


In [2]:
ext = pd.read_csv(CF_DATA / 'extinction.csv', dtype={'source_id': 'Int64'})
print(f"Loaded extinction.csv: {len(ext)} rows")
print(f"  A_Ks range: {ext['A_Ks'].min():.4f} – {ext['A_Ks'].max():.4f}")
print(f"  Stars with NaN A_Ks: {ext['A_Ks'].isna().sum()}")

Loaded extinction.csv: 11021 rows
  A_Ks range: 0.0000 – 0.2344
  Stars with NaN A_Ks: 130


## Collect all unique Gaia DR3 source IDs

In [3]:
def get_gaia_id_col(df):
    for col in ['source_id', 'GaiaDR3_ID']:
        if col in df.columns:
            return col
    return None

def get_ids(df):
    col = get_gaia_id_col(df)
    return df[col].dropna().astype(int).unique().tolist() if col else []

train_ids = get_ids(train)
val_ids   = get_ids(val)
mm_ids    = get_ids(mm)
cfrc_ids  = get_ids(cfrc)

all_ids = list(set(train_ids + val_ids + mm_ids + cfrc_ids))
print(f"Unique Gaia DR3 IDs across all catalogs: {len(all_ids)}")

Unique Gaia DR3 IDs across all catalogs: 11021


In [4]:
# Check TAP endpoint availability
import requests
for name, url in [('ARI Heidelberg', 'https://gaia.ari.uni-heidelberg.de/tap/tables'),
                  ('ESA Gaia',       'https://gea.esac.esa.int/tap-server/tap/tables')]:
    try:
        r = requests.get(url, timeout=10)
        print(f'{name}: {r.status_code}')
    except Exception as e:
        print(f'{name}: UNREACHABLE ({e})')

ARI Heidelberg: 200
ESA Gaia: UNREACHABLE (HTTPSConnectionPool(host='gea.esac.esa.int', port=443): Read timed out. (read timeout=10))


In [5]:
import requests
import time

def chunk_list(lst, n=500):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def query_vizier_tmass(designations, retries=3, delay=20):
    """Query CDS TAPVizieR for 2MASS Ks photometry by designation list."""
    url = 'http://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync'
    desig_str = ','.join(f"'{d.strip()}'" for d in designations)
    adql = f"""
        SELECT "2MASS" AS twomass_id, "Kmag" AS k_m, "e_Kmag" AS k_cmsig
        FROM "II/246/out"
        WHERE "2MASS" IN ({desig_str})
    """
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(url,
                data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'json','QUERY':adql},
                timeout=120)
            r.raise_for_status()
            d = r.json()
            cols = [c['name'] for c in d['metadata']]
            df = pd.DataFrame(d['data'], columns=cols)
            df['twomass_id'] = df['twomass_id'].str.strip()
            return df
        except Exception as e:
            if attempt < retries:
                print(f"    VizieR attempt {attempt} failed ({e}), retrying in {delay}s...")
                time.sleep(delay)
            else:
                raise

print('Helper functions defined.')

Helper functions defined.


In [6]:
# Primary 2MASS crossmatch via Gaia DR3 cross-match table (tmass_psc_xsc_best_neighbour)
#
# Step 1: Get 2MASS designations from ARI Heidelberg (sync POST, fast)
# Step 2: Fetch Ks photometry from CDS TAPVizieR (II/246/out) using those designations

ARI_SYNC = 'https://gaia.ari.uni-heidelberg.de/tap/sync'

def query_2mass_designations_ari(source_ids, retries=3, delay=20):
    """Get 2MASS designations for a list of Gaia DR3 source_ids via ARI sync TAP."""
    ids_str = ','.join(str(x) for x in source_ids)
    query = f"""
    SELECT source_id, original_ext_source_id AS twomass_id
    FROM gaiadr3.tmass_psc_xsc_best_neighbour
    WHERE source_id IN ({ids_str})
    """
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(ARI_SYNC,
                data={'REQUEST': 'doQuery', 'LANG': 'ADQL', 'FORMAT': 'json', 'QUERY': query},
                timeout=120)
            r.raise_for_status()
            d = r.json()
            cols = [c['name'] for c in d['metadata']]
            return pd.DataFrame(d['data'], columns=cols)
        except Exception as e:
            if attempt < retries:
                print(f'    ARI attempt {attempt} failed ({e}), retrying in {delay}s...')
                time.sleep(delay)
            else:
                print(f'    ARI all {retries} attempts failed — skipping chunk')
                return pd.DataFrame(columns=['source_id', 'twomass_id'])

# Step 1: get designations from ARI
desig_parts = []
chunks = list(chunk_list(all_ids, n=500))
for i, chunk in enumerate(chunks, start=1):
    print(f'Chunk {i}/{len(chunks)}: {len(chunk)} IDs', end=' ', flush=True)
    part = query_2mass_designations_ari(chunk)
    print(f'\u2192 {len(part)} designations')
    desig_parts.append(part)
    time.sleep(0.5)

desig_df = pd.concat(desig_parts, ignore_index=True)
desig_df['source_id'] = desig_df['source_id'].astype('Int64')
desig_df['twomass_id'] = desig_df['twomass_id'].str.strip()
print(f'\nTotal designations: {len(desig_df)} for {desig_df["source_id"].nunique()} unique source_ids')

# Step 2: fetch Ks photometry from CDS VizieR by designation
all_designations = desig_df['twomass_id'].dropna().unique().tolist()
print(f'Fetching photometry for {len(all_designations)} designations from CDS...')
phot_parts = []
for i, chunk in enumerate(chunk_list(all_designations, n=500), start=1):
    print(f'  Photometry chunk {i}: {len(chunk)} designations')
    phot_parts.append(query_vizier_tmass(chunk))
    time.sleep(0.5)

phot_df = pd.concat(phot_parts, ignore_index=True)
phot_df['twomass_id'] = phot_df['twomass_id'].str.strip()
print(f'Photometry returned for {len(phot_df)} designations')

# Join designations + photometry → tmass table keyed by source_id
tmass = desig_df.merge(phot_df, on='twomass_id', how='left')
print(f'\n2MASS matches with photometry: {tmass["k_m"].notna().sum()} / {len(tmass)}')
print(f'Unique source IDs matched: {tmass["source_id"].nunique()}')
tmass.head()

Chunk 1/23: 500 IDs → 447 designations
Chunk 2/23: 500 IDs → 452 designations
Chunk 3/23: 500 IDs → 466 designations
Chunk 4/23: 500 IDs → 474 designations
Chunk 5/23: 500 IDs → 464 designations
Chunk 6/23: 500 IDs → 470 designations
Chunk 7/23: 500 IDs → 460 designations
Chunk 8/23: 500 IDs → 462 designations
Chunk 9/23: 500 IDs → 460 designations
Chunk 10/23: 500 IDs → 461 designations
Chunk 11/23: 500 IDs → 464 designations
Chunk 12/23: 500 IDs → 463 designations
Chunk 13/23: 500 IDs → 460 designations
Chunk 14/23: 500 IDs → 465 designations
Chunk 15/23: 500 IDs → 470 designations
Chunk 16/23: 500 IDs → 467 designations
Chunk 17/23: 500 IDs → 461 designations
Chunk 18/23: 500 IDs → 466 designations
Chunk 19/23: 500 IDs → 463 designations
Chunk 20/23: 500 IDs → 468 designations
Chunk 21/23: 500 IDs → 469 designations
Chunk 22/23: 500 IDs → 475 designations
Chunk 23/23: 21 IDs → 19 designations

Total designations: 10226 for 10226 unique source_ids
Fetching photometry for 10225 design

,source_id,twomass_id,k_m,k_cmsig
0,45968901826385024,04081110+1652229,7.933,0.024
1,50856471531291648,03551362+1930170,11.721,0.021
2,51637193507243136,03525834+2105203,12.548,0.023
3,53105728725582976,04065602+2202049,10.833,0.020
4,63824901943723392,03520437+2145392,13.292,0.033


In [7]:
# Fallback: for stars with '2MASS J...' names that weren't matched via the Gaia cross-match
# (e.g. Mamonova2025, LWRD stars whose source_ids are float64-rounded and invalid),
# query gaiadr1.tmass_original_valid directly by 2MASS designation.

def extract_2mass_designation(name):
    """Convert '2MASS J00125703-7952073' -> '00125703-7952073'"""
    if isinstance(name, str) and name.startswith('2MASS J'):
        return name[7:]  # strip '2MASS J'
    return None

# Collect all star_name -> source_id mappings that need fallback
# (any catalog except cfrc, since cfrc uses GaiaDR3_ID not star_name)
all_nongaia_cats = pd.concat([train, val, mm], ignore_index=True)
unmatched_mask = ~all_nongaia_cats['source_id'].isin(tmass['source_id'])
unmatched_stars = all_nongaia_cats[unmatched_mask].copy()
unmatched_stars['designation'] = unmatched_stars['star_name'].apply(extract_2mass_designation)
to_lookup = unmatched_stars[unmatched_stars['designation'].notna()][['star_name','source_id','designation']].drop_duplicates('designation')
print(f"Stars to look up by 2MASS designation: {len(to_lookup)}")

# Query in chunks by designation
def query_2mass_by_designation(designations):
    return query_vizier_tmass(designations)

desig_parts = []
desig_list = to_lookup['designation'].tolist()
for i, chunk in enumerate(chunk_list(desig_list, n=500), start=1):
    print(f"Designation chunk {i}: {len(chunk)}")
    desig_parts.append(query_2mass_by_designation(chunk))

tmass_desig = pd.concat(desig_parts, ignore_index=True)
print(f"Found {len(tmass_desig)} matches by designation")

# Map back to source_id via designation -> star_name -> source_id
desig_to_sid = to_lookup.set_index('designation')['source_id'].to_dict()
tmass_desig['source_id'] = tmass_desig['twomass_id'].map(desig_to_sid).astype('Int64')

# Append to main tmass table (avoiding duplicates)
existing_ids = set(tmass['source_id'].dropna())
new_rows = tmass_desig[~tmass_desig['source_id'].isin(existing_ids)]
tmass = pd.concat([tmass, new_rows], ignore_index=True)
print(f"tmass table now has {len(tmass)} rows ({len(new_rows)} added from designation fallback)")


Stars to look up by 2MASS designation: 26
Designation chunk 1: 26
Found 26 matches by designation
tmass table now has 10252 rows (26 added from designation fallback)


In [8]:
import mk_mass

MANN_MKS_MIN = 4.5
MANN_MKS_MAX = 10.5

def compute_MKs(k_m_0, dist_pc):
    return k_m_0 + 5 - 5 * np.log10(dist_pc)

def get_dist_and_err(row):
    plx = row.get('parallax', np.nan)
    plx_err = row.get('parallax_error', np.nan)
    if pd.isna(plx) or plx <= 0:
        return np.nan, np.nan
    dist = 1000.0 / plx
    edist = (1000.0 / plx**2) * plx_err if pd.notna(plx_err) else 0.0
    return dist, edist

def add_2mass_and_mass(df, id_col):
    df = df.copy()
    df[id_col] = df[id_col].astype('Int64')

    # Drop any pre-existing columns to avoid merge collisions on re-run
    stale = [c for c in ['twomass_id', 'k_m', 'k_cmsig', 'A_Ks', 'A_Ks_err',
                          'k_m_0', 'M_Ks', 'flag_outside_mann_range', 'mass_msun', 'mass_msun_err']
             if c in df.columns]
    if stale:
        df = df.drop(columns=stale)

    # Merge 2MASS photometry
    df = df.merge(tmass, how='left', left_on=id_col, right_on='source_id')
    if 'source_id' in df.columns and id_col != 'source_id':
        df = df.drop(columns=['source_id'])

    # Merge extinction (A_Ks, A_Ks_err) keyed by source_id
    df = df.merge(ext[['source_id', 'A_Ks', 'A_Ks_err']], how='left',
                  left_on=id_col, right_on='source_id')
    if 'source_id' in df.columns and id_col != 'source_id':
        df = df.drop(columns=['source_id'])

    # Dereddened apparent magnitude: k_m_0 = k_m - A_Ks (treat NaN A_Ks as 0)
    A_Ks_safe = df['A_Ks'].fillna(0.0)
    df['k_m_0'] = df['k_m'] - A_Ks_safe

    def row_MKs(row):
        if pd.isna(row['k_m_0']):
            return np.nan
        dist, _ = get_dist_and_err(row)
        if pd.isna(dist) or dist <= 0:
            return np.nan
        return compute_MKs(row['k_m_0'], dist)

    df['M_Ks'] = df.apply(row_MKs, axis=1)
    df['flag_outside_mann_range'] = df['M_Ks'].apply(
        lambda m: (m < MANN_MKS_MIN or m > MANN_MKS_MAX) if pd.notna(m) else None
    )

    def get_mass(row):
        if pd.isna(row['k_m_0']) or pd.isna(row['M_Ks']):
            return np.nan, np.nan
        if row['flag_outside_mann_range']:
            return np.nan, np.nan
        dist, edist = get_dist_and_err(row)
        if pd.isna(dist) or dist <= 0:
            return np.nan, np.nan
        ek_phot = row['k_cmsig'] if pd.notna(row.get('k_cmsig')) else 0.0
        ek_ext  = row['A_Ks_err'] if pd.notna(row.get('A_Ks_err')) else 0.0
        ek = np.sqrt(ek_phot**2 + ek_ext**2)
        try:
            result = mk_mass.posterior(K=row['k_m_0'], dist=dist, ek=ek, edist=edist, oned=True, silent=True)
            return result[0], result[1]
        except Exception:
            return np.nan, np.nan

    print(f"Computing masses for {len(df)} rows...")
    mass_results = df.apply(get_mass, axis=1)
    df['mass_msun']     = [r[0] for r in mass_results]
    df['mass_msun_err'] = [r[1] for r in mass_results]

    return df

print("Functions defined.")

Functions defined.


In [9]:
train_out = add_2mass_and_mass(train, 'source_id')
val_out   = add_2mass_and_mass(val,   'source_id')
mm_out    = add_2mass_and_mass(mm,    'source_id')
cfrc_out  = add_2mass_and_mass(cfrc,  'GaiaDR3_ID')

for name, df in [('training_stars', train_out), ('validation_stars', val_out),
                 ('mmcoeval', mm_out), ('cf_rotator_catalog', cfrc_out)]:
    matched   = df['k_m'].notna().sum()
    in_range  = df['flag_outside_mann_range'].eq(False).sum()
    outside   = df['flag_outside_mann_range'].eq(True).sum()
    with_mass = df['mass_msun'].notna().sum()
    print(f"{name}: {matched}/{len(df)} 2MASS matched | in Mann range: {in_range} | outside: {outside} | mass computed: {with_mass}")

Computing masses for 4923 rows...
0.06% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation


/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


0.02% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.02% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.36% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.00% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
0.05% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.02% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.03% of posterior is outside suggested range for the relation
0.04% of posterior is outside suggested range for the relation
0.05% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.24% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
Warning, some outputs are Na

/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


0.05% of posterior is outside suggested range for the relation
2.94% of posterior is outside suggested range for the relation
0.42% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.47% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
1.06% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
14.13% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
0.02% of posterior is outside suggested range for the relation
0.08% of posterior is outside suggested range for the relation
1.24% of posterior is outside suggested range for the relation
0.00% of posterior is outside suggested range for the relation
15.14% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
5.64% of posterior is outside suggested range for the relation
0.03% of posterior is outside suggested range for the relation

In [10]:
# SIMBAD fallback: resolve unmatched stars to 2MASS designation via SIMBAD,
# then fetch photometry from CDS TAPVizieR (independent of Gaia archive)

from astroquery.simbad import Simbad

simbad = Simbad()
simbad.add_votable_fields('ids')

def extract_2mass_from_ids(ids_str):
    if not ids_str:
        return None
    for ident in str(ids_str).split('|'):
        ident = ident.strip()
        if ident.startswith('2MASS J'):
            return ident[7:]
    return None

def get_2mass_designations_simbad(df, id_col):
    unmatched = df[df['k_m'].isna()].copy()
    if len(unmatched) == 0:
        return {}
    name_col = 'star_name' if 'star_name' in df.columns else None

    queries = {}  # idx -> query string
    for idx, row in unmatched.iterrows():
        sname = str(row[name_col]) if name_col and pd.notna(row.get(name_col)) else None
        sid   = row.get(id_col)
        if sname and not sname.startswith('Gaia DR'):
            queries[idx] = sname
        elif pd.notna(sid):
            queries[idx] = f'Gaia DR3 {int(sid)}'

    print(f'  Querying SIMBAD for {len(queries)} stars...')
    query_list = list(queries.values())
    try:
        result = simbad.query_objects(query_list)
    except Exception as e:
        print(f'  SIMBAD query failed: {e}')
        return {}
    if result is None:
        print('  SIMBAD returned no results')
        return {}

    desig_map = {}
    idx_list = list(queries.keys())
    for pos, row in enumerate(result):
        if pos >= len(idx_list):
            break
        ids_str = str(row['ids']) if row['ids'] is not None else ''
        desig = extract_2mass_from_ids(ids_str)
        if desig:
            desig_map[idx_list[pos]] = desig

    print(f'  Resolved 2MASS designations for {len(desig_map)}/{len(queries)} stars')
    return desig_map

def apply_simbad_fallback(df, id_col):
    df = df.copy()
    desig_map = get_2mass_designations_simbad(df, id_col)
    if not desig_map:
        return df

    designations = list(set(desig_map.values()))
    tmass_new = query_vizier_tmass(designations).set_index('twomass_id')
    print(f'  2MASS returned {len(tmass_new)} rows for {len(designations)} designations')

    filled = 0
    for idx, desig in desig_map.items():
        if desig not in tmass_new.index:
            continue
        k_m     = float(tmass_new.loc[desig, 'k_m'])
        k_cmsig = float(tmass_new.loc[desig, 'k_cmsig']) if pd.notna(tmass_new.loc[desig, 'k_cmsig']) else 0.0
        df.at[idx, 'twomass_id'] = desig
        df.at[idx, 'k_m']        = k_m
        df.at[idx, 'k_cmsig']    = k_cmsig

        # Apply extinction correction (A_Ks already merged by add_2mass_and_mass)
        A_Ks     = df.at[idx, 'A_Ks']     if pd.notna(df.at[idx, 'A_Ks'])     else 0.0
        A_Ks_err = df.at[idx, 'A_Ks_err'] if pd.notna(df.at[idx, 'A_Ks_err']) else 0.0
        k_m_0 = k_m - A_Ks
        df.at[idx, 'k_m_0'] = k_m_0

        dist, edist = get_dist_and_err(df.loc[idx])
        if pd.notna(dist) and dist > 0:
            mks = compute_MKs(k_m_0, dist)
            df.at[idx, 'M_Ks'] = mks
            outside = (mks < MANN_MKS_MIN or mks > MANN_MKS_MAX)
            df.at[idx, 'flag_outside_mann_range'] = outside
            if not outside:
                ek = np.sqrt(k_cmsig**2 + A_Ks_err**2)
                try:
                    res = mk_mass.posterior(K=k_m_0, dist=dist, ek=ek,
                                           edist=edist, oned=True, silent=True)
                    df.at[idx, 'mass_msun']     = res[0]
                    df.at[idx, 'mass_msun_err'] = res[1]
                except Exception:
                    pass
        filled += 1
    print(f'  Filled {filled} rows')
    return df

print('Applying SIMBAD fallback to training_stars...')
train_out = apply_simbad_fallback(train_out, 'source_id')
print('Applying SIMBAD fallback to validation_stars...')
val_out   = apply_simbad_fallback(val_out,   'source_id')
print('Applying SIMBAD fallback to mmcoeval...')
mm_out    = apply_simbad_fallback(mm_out,    'source_id')

print()
for name, df in [('training_stars', train_out), ('validation_stars', val_out), ('mmcoeval', mm_out)]:
    print(f'{name}: {df["k_m"].notna().sum()}/{len(df)} 2MASS matched | mass: {df["mass_msun"].notna().sum()}')

Applying SIMBAD fallback to training_stars...
  Querying SIMBAD for 201 stars...
  Resolved 2MASS designations for 44/201 stars
  2MASS returned 44 rows for 44 designations
  Filled 44 rows
Applying SIMBAD fallback to validation_stars...
  Querying SIMBAD for 122 stars...
  Resolved 2MASS designations for 116/122 stars
  2MASS returned 116 rows for 116 designations
  Filled 116 rows
Applying SIMBAD fallback to mmcoeval...
  Querying SIMBAD for 12 stars...
  Resolved 2MASS designations for 12/12 stars
  2MASS returned 12 rows for 12 designations
  Filled 12 rows

training_stars: 4766/4923 2MASS matched | mass: 3987
validation_stars: 236/242 2MASS matched | mass: 223
mmcoeval: 34/34 2MASS matched | mass: 33


In [27]:
# MOCADB + positional fallback: recover unmatched M dwarfs
#
# 1. Pull twomass_id directly from mocadb.csv for MOCADB stars that have one
# 2. Fetch pmra/pmdec from Gaia DR3, epoch-correct J2016→J2000, CDS XMatch for designations
# 3. Use the 2MASS RAJ2000/DEJ2000 from XMatch result to do a 1" cone per star for Kmag

from astroquery.xmatch import XMatch
from astroquery.vizier import Vizier as VizQ
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

moca_src = pd.read_csv(CF_DATA / 'mocadb.csv', dtype={'GaiaDR3_ID': 'Int64'})
moca_tid  = moca_src[['GaiaDR3_ID', 'twomass_id']].dropna(subset=['twomass_id'])
moca_tid_renamed = moca_tid.rename(columns={'twomass_id': 'moca_twomass_id'})

vq = VizQ(row_limit=1)

def fetch_gaia_pm(source_ids):
    ids_str = ','.join(str(x) for x in source_ids)
    query = f"""SELECT source_id, pmra, pmdec FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})"""
    r = requests.post(ARI_SYNC,
        data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'json','QUERY':query}, timeout=60)
    r.raise_for_status()
    d = r.json()
    df_pm = pd.DataFrame(d['data'], columns=[c['name'] for c in d['metadata']])
    df_pm['source_id'] = df_pm['source_id'].astype('Int64')
    return df_pm

def epoch_correct_j2016_to_j2000(ra, dec, pmra, pmdec):
    dt = -16.0
    dec_rad = np.radians(dec)
    return (ra  + pmra  * dt / (np.cos(dec_rad) * 3.6e6),
            dec + pmdec * dt / 3.6e6)

def fill_row(df, idx, twomass_id, k_m, k_cmsig):
    A_Ks     = df.at[idx, 'A_Ks']     if pd.notna(df.at[idx, 'A_Ks'])     else 0.0
    A_Ks_err = df.at[idx, 'A_Ks_err'] if pd.notna(df.at[idx, 'A_Ks_err']) else 0.0
    k_m_0 = k_m - A_Ks
    df.at[idx, 'twomass_id'] = twomass_id
    df.at[idx, 'k_m']        = k_m
    df.at[idx, 'k_cmsig']    = k_cmsig
    df.at[idx, 'k_m_0']      = k_m_0
    dist, edist = get_dist_and_err(df.loc[idx])
    if pd.notna(dist) and dist > 0:
        mks = compute_MKs(k_m_0, dist)
        df.at[idx, 'M_Ks'] = mks
        outside = (mks < MANN_MKS_MIN or mks > MANN_MKS_MAX)
        df.at[idx, 'flag_outside_mann_range'] = outside
        if not outside:
            ek = np.sqrt(k_cmsig**2 + A_Ks_err**2)
            try:
                res = mk_mass.posterior(K=k_m_0, dist=dist, ek=ek,
                                        edist=edist, oned=True, silent=True)
                df.at[idx, 'mass_msun']     = res[0]
                df.at[idx, 'mass_msun_err'] = res[1]
            except Exception:
                pass

def patch_with_twomass(df, id_col):
    df = df.copy()
    unmatched_idx = df[df['k_m'].isna()].index
    if len(unmatched_idx) == 0:
        print('  No unmatched rows.')
        return df

    filled = 0

    # --- Step 1: mocadb.csv direct lookup ---
    moca_hits = df.loc[unmatched_idx].merge(
        moca_tid_renamed, how='inner', left_on=id_col, right_on='GaiaDR3_ID')
    moca_resolved_idx = set(moca_hits.index)
    if len(moca_hits) > 0:
        desigs = moca_hits['moca_twomass_id'].str.strip().unique().tolist()
        print(f'  mocadb direct lookup: {len(desigs)} designations')
        phot = query_vizier_tmass(desigs).set_index('twomass_id')
        for idx, row in moca_hits.iterrows():
            desig = row['moca_twomass_id'].strip()
            if desig in phot.index:
                k_m = float(phot.loc[desig, 'k_m'])
                k_cmsig = float(phot.loc[desig, 'k_cmsig']) if pd.notna(phot.loc[desig, 'k_cmsig']) else 0.0
                fill_row(df, idx, desig, k_m, k_cmsig)
                filled += 1
    else:
        print('  mocadb direct lookup: 0 designations')

    # --- Step 2: epoch-correct + XMatch for remaining ---
    xmatch_candidates = df.loc[
        [i for i in unmatched_idx
         if i not in moca_resolved_idx
         and pd.notna(df.at[i, 'ra']) and pd.notna(df.at[i, 'dec'])],
        [id_col, 'ra', 'dec']
    ].copy()

    if len(xmatch_candidates) == 0:
        print(f'  Filled {filled} rows')
        return df

    print(f'  Fetching proper motions for {len(xmatch_candidates)} stars...')
    pm_df = fetch_gaia_pm(xmatch_candidates[id_col].dropna().astype(int).tolist())

    # Use .map() to preserve xmatch_candidates index (merge resets it)
    pm_indexed = pm_df.set_index('source_id')
    xmatch_candidates['pmra']  = xmatch_candidates[id_col].map(pm_indexed['pmra'])
    xmatch_candidates['pmdec'] = xmatch_candidates[id_col].map(pm_indexed['pmdec'])
    pmra_  = xmatch_candidates['pmra'].fillna(0.0)
    pmdec_ = xmatch_candidates['pmdec'].fillna(0.0)

    ra_corr, dec_corr = epoch_correct_j2016_to_j2000(
        xmatch_candidates['ra'].values, xmatch_candidates['dec'].values,
        pmra_.values, pmdec_.values)
    xmatch_candidates['ra_j2000']  = ra_corr
    xmatch_candidates['dec_j2000'] = dec_corr
    print(f'  Epoch-corrected {(pmra_.abs()+pmdec_.abs()>0).sum()}/{len(xmatch_candidates)} stars')

    print(f'  CDS XMatch: {len(xmatch_candidates)} stars at 5 arcsec...')
    t = Table.from_pandas(
        xmatch_candidates.reset_index().rename(columns={'index':'_orig_idx'})
        [['_orig_idx','ra_j2000','dec_j2000']])
    try:
        result = XMatch.query(cat1=t, cat2='vizier:II/246/out',
                              max_distance=5*u.arcsec,
                              colRA1='ra_j2000', colDec1='dec_j2000')
        result_df = result.to_pandas()
        # Keep closest match per star
        result_df['_orig_idx'] = result_df['_orig_idx'].astype(int)
        result_df = result_df.sort_values('angDist').drop_duplicates('_orig_idx')
        print(f'  XMatch returned {len(result_df)} matches — fetching Kmag via 1" cone...')

        for _, row in result_df.iterrows():
            idx = int(row['_orig_idx'])
            if pd.notna(df.at[idx, 'k_m']):
                continue
            ra_tmass  = float(row['RAJ2000'])
            dec_tmass = float(row['DEJ2000'])
            twomass_id = str(row['2MASS']).strip()
            try:
                coord = SkyCoord(ra_tmass, dec_tmass, unit='deg')
                q = vq.query_region(coord, radius=1*u.arcsec, catalog='II/246')
                if q and 'II/246/out' in q.keys() and len(q['II/246/out']) > 0:
                    r = q['II/246/out'].to_pandas()
                    if pd.notna(r.iloc[0].get('Kmag')):
                        k_m     = float(r.iloc[0]['Kmag'])
                        k_cmsig = float(r.iloc[0]['e_Kmag']) if pd.notna(r.iloc[0].get('e_Kmag')) else 0.0
                        fill_row(df, idx, twomass_id, k_m, k_cmsig)
                        filled += 1
            except Exception as e:
                pass
    except Exception as e:
        print(f'  XMatch failed: {e}')

    print(f'  Filled {filled} rows')
    return df

print('Applying MOCADB + positional fallback to training_stars...')
train_out = patch_with_twomass(train_out, 'source_id')
print('Applying to validation_stars...')
val_out   = patch_with_twomass(val_out,   'source_id')
print('Applying to mmcoeval...')
mm_out    = patch_with_twomass(mm_out,    'source_id')

print()
for name, df in [('training_stars', train_out), ('validation_stars', val_out), ('mmcoeval', mm_out)]:
    print(f'{name}: k_m={df["k_m"].notna().sum()}/{len(df)} | mass={df["mass_msun"].notna().sum()}')

Applying MOCADB + positional fallback to training_stars...
  mocadb direct lookup: 0 designations
  Fetching proper motions for 101 stars...
  Epoch-corrected 98/101 stars
  CDS XMatch: 101 stars at 5 arcsec...
  XMatch returned 30 matches — fetching Kmag via 1" cone...
0.01% of posterior is outside suggested range for the relation
  Filled 30 rows
Applying to validation_stars...
  mocadb direct lookup: 0 designations
  Fetching proper motions for 5 stars...
  Epoch-corrected 0/5 stars
  CDS XMatch: 5 stars at 5 arcsec...
  XMatch returned 0 matches — fetching Kmag via 1" cone...
  Filled 0 rows
Applying to mmcoeval...
  No unmatched rows.

training_stars: k_m=4852/4923 | mass=4021
validation_stars: k_m=237/242 | mass=223
mmcoeval: k_m=34/34 | mass=33


In [28]:
# Run SIMBAD + positional fallback on cf_rotator_catalog
# (same functions as cells 11 & 12, just not applied to cfrc before)

print('Applying SIMBAD fallback to cf_rotator_catalog...')
cfrc_out = apply_simbad_fallback(cfrc_out, 'GaiaDR3_ID')

print('Applying MOCADB + positional fallback to cf_rotator_catalog...')
cfrc_out = patch_with_twomass(cfrc_out, 'GaiaDR3_ID')

print()
n_km   = cfrc_out['k_m'].notna().sum()
n_mass = cfrc_out['mass_msun'].notna().sum()
n_tot  = len(cfrc_out)
print(f'cf_rotator_catalog: k_m={n_km}/{n_tot} | mass={n_mass}/{n_tot}')
print(f'Still no k_m: {n_tot - n_km}')

Applying SIMBAD fallback to cf_rotator_catalog...
  Querying SIMBAD for 432 stars...
  Resolved 2MASS designations for 0/432 stars
Applying MOCADB + positional fallback to cf_rotator_catalog...
  mocadb direct lookup: 0 designations
  Fetching proper motions for 432 stars...
  Epoch-corrected 432/432 stars
  CDS XMatch: 432 stars at 5 arcsec...
  XMatch returned 54 matches — fetching Kmag via 1" cone...


/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


10.49% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!


/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


1.65% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!


...

33.24% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!


/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


6.81% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!


/Users/alanxia/Documents/GitHub/cf_xmatch/Mann_2019/mk_mass.py:129: RuntimeWarning: invalid value encountered in log10
  mk       = kmag + 5 - 5*np.log10(distance)


3.42% of posterior is outside suggested range for the relation
Warning, some outputs are NaNs!!?!!
  Filled 54 rows

cf_rotator_catalog: k_m=5454/5832 | mass=2125/5832
Still no k_m: 378


## Save outputs

In [29]:
train_out.to_csv(CF_DATA / "training_stars.csv", index=False)
val_out.to_csv(CF_DATA / "validation_stars.csv", index=False)
mm_out.to_csv(CF_DATA / "mmcoeval.csv", index=False)
cfrc_out.to_csv(CF_DATA / "cf_rotator_catalog.csv", index=False)

print("Saved all four files.")

Saved all four files.


In [30]:
# Save stars without 2MASS Ks to a separate flagged list

dfs = []
for name, df, id_col in [
    ('training_stars',    train_out, 'source_id'),
    ('validation_stars',  val_out,   'source_id'),
    ('mmcoeval',          mm_out,    'source_id'),
    ('cf_rotator_catalog',cfrc_out,  'GaiaDR3_ID'),
]:
    unmatched = df[df['k_m'].isna()].copy()
    unmatched['catalog'] = name
    unmatched['gaia_id'] = unmatched[id_col]
    keep = ['catalog', 'gaia_id']
    for c in ['star_name', 'ra', 'dec', 'parallax', 'source_paper']:
        if c in unmatched.columns:
            keep.append(c)
    dfs.append(unmatched[keep])

flagged = pd.concat(dfs, ignore_index=True)
flagged.to_csv(CF_DATA / 'no_ks_flagged.csv', index=False)
print(f'Saved {len(flagged)} stars without Ks to cf_data/no_ks_flagged.csv')
print(flagged['catalog'].value_counts())

Saved 454 stars without Ks to cf_data/no_ks_flagged.csv
catalog
cf_rotator_catalog    378
training_stars         71
validation_stars        5
Name: count, dtype: int64
